## Energy Insights and Optimization Recommendations

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
model_df= pd.read_csv("C:/EcoWatt-AI/data/processed/energy_with_anomalies.csv",parse_dates=["datetime"])

In [ ]:
model_df.info()

In [ ]:
model_df.columns

In [ ]:
print("Average Power:",
      model_df["Global_active_power"].mean())

print("Maximum Power:",
      model_df["Global_active_power"].max())

print("Minimum Power:",
      model_df["Global_active_power"].min())

print("Median Power:",
      model_df["Global_active_power"].median())

## Daily Energy Profle

In [ ]:
model_df["date"] = model_df["datetime"].dt.date

daily_energy = (
    model_df.groupby("date")["Global_active_power"]
    .mean()
)

In [ ]:
plt.figure(figsize=(18,5))

daily_energy.plot()

plt.title("Daily Average Energy Consumption")

plt.ylabel("Average Active Power (kW)")

plt.tight_layout()

plt.show()

In [ ]:
hourly_power = (
    model_df.groupby("hour")["Global_active_power"]
    .mean()
)

## Hourly Consumption Profile

In [ ]:
plt.figure(figsize=(12,5))

hourly_power.plot(marker="o")

plt.title("Average Hourly Energy Consumption")

plt.xlabel("Hour")

plt.ylabel("Average Active Power")

plt.grid(True)

plt.show()

In [ ]:
top_hours = (
    hourly_power
    .sort_values(ascending=False)
)

print(top_hours.head(10))

 Weekend vs Weekday Analysis

In [ ]:
model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
)

In [ ]:
week_compare = (
    model_df
    .groupby("is_weekend")["Global_active_power"]
    .mean()
)

print(week_compare)

Anomaly Power consumption

In [ ]:
comparison = (
    model_df.groupby("anomaly")[
        "Global_active_power"
    ].mean()
)

print(comparison)

## Estimate Energy Wastage

In [ ]:
normal_mean = (
    model_df.loc[
        model_df["anomaly"]==0,
        "Global_active_power"
    ].mean()
)

In [ ]:
anomaly_df = model_df[
    model_df["anomaly"]==1
].copy()

anomaly_df["estimated_excess_power"] = (
    anomaly_df["Global_active_power"]
    - normal_mean
)

anomaly_df["estimated_excess_power"] = (
    anomaly_df["estimated_excess_power"]
    .clip(lower=0)
)

In [ ]:
print(
    anomaly_df[
        "estimated_excess_power"
    ].sum()
)

## Energy Optimization Recommendations

In [ ]:
def generate_recommendation(row):

    if row["Global_active_power"] > 4:
        return "Inspect high-power appliances."

    elif row["rolling_zscore_15_log"] > 2:
        return "Investigate sudden consumption spike."

    elif row["active_power_change_rate_log"] > 2:
        return "Check recently activated electrical devices."

    elif row["Voltage"] < 220:
        return "Inspect supply voltage stability."

    else:
        return "Monitor household consumption."

In [ ]:
anomaly_df["recommendation"] = (
    anomaly_df.apply(
        generate_recommendation,
        axis=1
    )
)

In [ ]:
anomaly_df[
    [
        "datetime",
        "Global_active_power",
        "recommendation"
    ]
].head(20)

In [ ]:
anomaly_df.to_csv("C:/EcoWatt-AI/data/processed/energy_insights.csv",index=False)